# 1.1 - 1.2 Intro - Env setup, (LLM API key, model def)

In [1]:
# install dirdotenv to automaticaly load .env files (load within each direcctory)
!uv tool install dirdotenv
# do it only once:
# !echo 'eval "$(dirdotenv hook bash)"' >> ~/.bashrc

`dirdotenv` is already installed


In [2]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# 1.3 RAG - Test the LLM directly with no context

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="llama-3.1-8b-instant", # modele chez groq equivalent
        # model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
llm("Hey, what's up?")

"Not much. I'm here to help with any questions or topics you'd like to discuss. What's been on your mind lately?"

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm happy to help you with your question. However, I'm a large language model, I don't have any information about a specific course you're referring to. You mentioned you just discovered the course, but I didn't get any details about it.

Could you please provide more context or details about the course you're interested in joining? This will help me assist you better.


## Add context from FAQ

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can join the course now. 

However, if you aim to receive a certificate upon completion, make sure to submit your project while the course is still accepting submissions.


OK, that's RAG (giving proper context to LLM)

## Retrieval plus generation

Goal: get something like that

In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# 1.4 Course FAQ Dataset 

Fetch directly FAQ (available as json format for ease of maintenance)

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
print(courses_raw)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 85}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


Fetch all FAQs for all courses

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

Each entry has:

- `id` - unique identifier
- `course` - course slug (e.g., `machine-learning-zoomcamp`)
- `section` - which section of the course
- `question` - the FAQ question
- `answer` - the FAQ answer

Let's look at one:

In [13]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

## **Using this data**

In the RAG pipeline, dataset = knowledge base:

1. We **index** all the documents (the search step)
2. When a student asks a question, we **search the index**
3. The search returns the **most relevant** FAQ **entries**
4. We **give** those **entries to** the **LLM** as **context**
5. The **LLM generates** an **answer** based on the context

- The `question` and `answer` fields contain the text we'll search through.  
- The `course` field lets us filter by course. 
- For example, if a student asks about the data engineering course, we skip results from the ML course. 
- The `section` field helps with ranking - knowing which part of the course a question belongs to is useful context.

## **Data preparation**

- here: data **already prepared** (convenient JSON format). 
- A lot of this data **cleaned** (with the help of an LLM, handy use case on its own).
- In reality, **data preparation** is often the **most time-consuming part** of building a RAG system. 
  - scrape websites, 
  - parse PDFs, 
  - clean documents
  - chunk documents.

# 1.5 Search

At its core, every search engine does the same thing. It takes a **query**, **scores** every document for **similarity**, and returns the **top results**.

Formally, there is a similarity function:

In [14]:
# score = sim(query, document)

Main **difference** between search engines: **sim function**

2 ways:
- **text/lexical** search (module 1): 
  - `sim` counts the **number of words** the **query** and the **document** **share**. 
  - It looks at the **surface form**, the **actual words**, and **matches them** exactly.
- **vector/semantic** search (module 2): 
  - `sim` compares the **meaning** of the **query** and the **document**. 
  - Same function, **different similarity** measure.

Consider these two questions:

- "Can I still join the course after the start date?"
- "Is it possible to enroll late?"

They mean the same thing, but share almost no keywords. "Join" is not "enroll", "course" is absent, "start date" is not "late". A **text search** engine would **struggle** to match them, because it **only sees words**.

We'll see how vector search solves this later. For now, let's build text search with minsearch.

## Create an index with minsearch

Let's index the `documents`.

 Sending 1100 documents to the LLM would be **expensive** and **slow**. The model would get confused with **too much data**. Search finds the most relevant documents to send instead.

There are **many** search **libraries** you can use - Apache Lucene, Elasticsearch, Solr, and others (heavy). 

**minsearch** is a **simple in-memory** search engine (lightweight, runs anywhere Python runs, including Google Colab). It's a **toy implementation**, not production ready, but it **illustrates** **how search engines work** and it gives good results.

The concepts :
  - same as in Elasticsearch (which comes from Lucene): 
      - text fields, 
      - keyword fields, 
      - boosting, 
      - filtering. 

We'll index as **text** (they'll be tokenized and ranked) the fields:
  - `question` 
  - `section` 
  - `answer` 
, and as a **keyword** (for filtering) the field
  - `course`

The index **tokenizes text fields** and treats **keyword fields** as **exact strings**.

**Text fields** :
  - the **fields you search through**. 
  - When you type a query :
    - the search engine looks at these fields 
    - and **tokenizes** them. 
    - It **splits text into words**, 
    - **lowercases** them, 
    - **removes stop words**, and 
    - **ranks the results** by relevance. 
  So `question`, `section`, and `answer` are text fields.

**Keyword fields** :
  - for **exact matching** (filtering). 
  - **restrict the search space** to a particular** subset**. 


In [15]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

## **Trying a search**

Let's try a search with the question we used before:

In [16]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [17]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

## Boosting fields


In [18]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

## Filtering by course

In [19]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [20]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

## Wrapping it in a function


In [21]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [22]:
search_results = search(question)


# 1.6 Building the Prompt

## Instructions

In [23]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

## The user prompt template

In [24]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

## Building the context

In [25]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

## Building the prompt

In [26]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [27]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

# 1.7 The LLM

## Sending the prompt to the LLM

In [28]:
# for openai 
# response = openai_client.responses.create(
#     model="meta-llama/llama-4-scout-17b-16e-instruct",
#     # model="gpt-5.4-mini",
#     input=prompt
# )

In [29]:
# for Groq
response = openai_client.responses.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    input=[
        {"role": "user", "content": prompt}
    ]
)

## Exploring the response

In [30]:
# openai
# response.output[0]

In [31]:
# 1. Récupérer les tokens envoyés (ton prompt)
prompt_tokens = response.usage.input_tokens  # <-- Remplacé par input_tokens

# 2. Récupérer les tokens générés par le modèle
completion_tokens = response.usage.output_tokens  # <-- Remplacé par output_tokens

# 3. Le total cumulé
total_tokens = response.usage.total_tokens

# --- Affichage ---
print(f"Tokens envoyés (Input)  : {prompt_tokens}")
print(f"Tokens générés (Output) : {completion_tokens}")
print(f"Total tokens            : {total_tokens}")

Tokens envoyés (Input)  : 475
Tokens générés (Output) : 243
Total tokens            : 718


## Calculating the price

In [32]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost


0.00144975

## Message history

In [33]:

message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    input=message_history
)




## The LLM function

In [34]:
def llm(instructions, user_prompt, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    response = openai_client.responses.create(
        model=model,
        instructions=instructions,  # Remplace le rôle "developer"
        input=user_prompt          # Remplace le rôle "user"
    )
    
    # N'oublie pas d'extraire le texte comme on l'a vu juste avant ;)
    return response.output_text

## Full RAG

In [35]:

def rag(query, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [36]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. The course materials are available, and you can start learning and submitting homework. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


## Try more questions

In [37]:

rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a "live" cohort. This means that you cannot get a certificate if you follow the course in a self-paced mode. Additionally, you need to pass the Capstone project. The homework is not mandatory, but it is recommended for reinforcing concepts. \n\nIn other words, to get a certificate, make sure to:\n- Finish the course with a live cohort\n- Pass the Capstone project (which includes peer-reviewing 3 capstones after submitting your project)'

# 1.8 RAG Helper 

## ingest.py

In [38]:
# import requests
# from minsearch import Index

# def load_faq_data():
#     docs_url = "https://datatalks.club/faq/json/courses.json"
#     response = requests.get(docs_url)
#     courses_raw = response.json()

#     documents = []
#     url_prefix = "https://datatalks.club/faq"

#     for course in courses_raw:
#         course_url = f"""{url_prefix}{course["path"]}"""
#         course_response = requests.get(course_url)
#         course_response.raise_for_status()
#         course_data = course_response.json()

#         documents.extend(course_data)

#     return documents

# def build_index(documents):
#     index = Index(
#         text_fields=["question", "section", "answer"],
#         keyword_fields=["course"]
#     )
#     index.fit(documents)
#     return index

## rag_helper.py

In [39]:
# INSTRUCTIONS = '''
# Your task is to answer questions from the course participants
# based on the provided context.

# Use the context to find relevant information and provide accurate
# answers. If the answer is not found in the context,
# respond with "I don't know."
# '''

# PROMPT_TEMPLATE = '''
# QUESTION: {question}

# CONTEXT:
# {context}
# '''.strip()


# class RAGBase:

#     def __init__(
#         self,
#         index,
#         llm_client,
#         instructions=INSTRUCTIONS,
#         prompt_template=PROMPT_TEMPLATE,
#         course='llm-zoomcamp',
#         model="meta-llama/llama-4-scout-17b-16e-instruct"
#         # model="llama-3.1-8b-instant"
#         # model='gpt-5.4-mini'
#     ):
#         self.index = index
#         self.llm_client = llm_client
#         self.instructions = instructions
#         self.course = course
#         self.prompt_template = prompt_template
#         self.model = model

#     def search(self, query, num_results=5):
#         boost_dict = {"question": 3.0, "section": 0.5}
#         filter_dict = {"course": self.course}

#         return self.index.search(
#             query,
#             num_results=num_results,
#             boost_dict=boost_dict,
#             filter_dict=filter_dict
#         )

#     def build_context(self, search_results):
#         lines = []

#         for doc in search_results:
#             lines.append(doc["section"])
#             lines.append("Q: " + doc["question"])
#             lines.append("A: " + doc["answer"])
#             lines.append("")

#         return "\n".join(lines).strip()

#     def build_prompt(self, query, search_results):
#         context = self.build_context(search_results)
#         return self.prompt_template.format(
#             question=query, context=context
#         )
    
#     def llm(self, prompt):
#         input_messages = [
#             {'role': 'developer', 'content': self.instructions},
#             {'role': 'user', 'content': prompt}
#         ]

#         response = self.llm_client.responses.create(
#             model=self.model,
#             input=input_messages
#         )

#         print("response.usage.input_tokens")
#         print(response.usage.input_tokens)

#         return response.output_text

#     def rag(self, query):
#         search_results = self.search(query)
#         prompt = self.build_prompt(query, search_results)
#         answer = self.llm(prompt)
#         return answer


## Using it in a notebook

In [40]:

# from dotenv import load_dotenv
# load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

response.usage.input_tokens
540
**Yes, you can join the course now.** 

The course materials, including videos and GitHub resources, are available, and you can start learning immediately. To get started:

1. **Check out the course documentation**: 
   - [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)
   - [General Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)
   - [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)

2. **Follow the weekly workflow**: 
   - Watch lesson videos
   - Work through lesson notebooks/code
   - Read homework instructions on GitHub
   - Submit answers through the course platform before the deadline

**Note on Certificate**: If you want to receive a certificate, you need to submit your project while submissions are still being accepted, and you must finish the course with a "live" cohort, which includes peer-reviewing 3 capstones. Self-paced mode does not qualify for a certifi

# 1.9 Data Ingestion

In [41]:
!uv add sqlitesearch

Resolved 129 packages in 1ms
Audited 125 packages in 8ms


## Ingestion notebook

In [42]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1350 documents


In [43]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 85 documents


In [44]:
import time
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: How should I start the course and follow the weekly workflow...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: Where can I track the LLM Zoomcamp syllabus, deadlines, home...
Added: Are there live sessions or office hours for each module?...
Added: Can I use Bluesky for learning in public credits?...
Added: Where is the LLM Zoomcamp Telegram channel?...
Added: Why are we not using Langchain in the course?...
Added: OpenAI: Error when running OpenAI respon

## Querying notebook

In [45]:

from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [46]:
sqlite_index.count()

425

In [47]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?']

## RAG with sqlitesearch

In [48]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [49]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

response.usage.input_tokens
250
Based on the provided context, the answer to your question is:

**Yes**, you can still join the course after it started. However, **if you want to receive a certificate**, you need to submit your project while submissions are still being accepted.


## Comparing the two approaches

With minsearch (single process):

```
Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps
```

With sqlitesearch (two processes):

```
Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready
```

## Choosing an approach

Pick the right tool for your data:

- `minsearch`: single process, in-memory only, re-indexes on every startup. Use when data is small and indexing is fast.
- `sqlitesearch`: separate ingestion and query, file-based (SQLite), opens existing index. Use when data is large or ingestion is slow.

Use minsearch when you can load and index the data on startup without noticeable delay. Switch to a persistent backend when ingestion takes too long or when you need the index to survive restarts.

For larger production systems, use the same pattern with a different backend:

- Elasticsearch
- OpenSearch
- Qdrant (vector database)
- Weaviate (vector database)

The architecture stays the same: one process ingests, another queries.

## Cleaning up

In [50]:
sqlite_index.close()

# 1.10 Wrap-up of Part 1

In Part 1 of this module, we:

- Learned what RAG is and why it matters: retrieve documents, build a prompt, let the LLM generate a grounded answer
- Built a search engine over a real FAQ dataset using minsearch
- Created a prompt template that combines the user's question with search results
- Wired search + prompt + LLM into a working RAG pipeline
- Split ingestion and query into separate processes with sqlitesearch

You now have a working RAG system and a clear mental model for how each piece fits together. From here, the work is making each piece better.

## Two directions forward

Part 2 of this module: Agents. Our pipeline runs search once with the exact user query, unchanged. If that search returns garbage, the LLM has no way to recover. An agent puts the LLM in charge instead. It decides what to search for, how many searches to run, and when to stop.

An agent also handles questions in another language. It translates the query before searching, then translates the answer back afterward.

Module 2: Vector Search. Keyword matches are exact. Vector search matches by semantic meaning instead, which helps when the user phrases things differently from the FAQ.

## Elasticsearch

Elasticsearch is the industry standard for text search.

It supports:

- Full-text search with BM25
- Filtering, aggregations, and faceting
- Vector search (dense and sparse)
- Distributed scaling
- Real-time indexing

It's heavier than sqlitesearch but handles production workloads at scale. If you're building a real RAG system, Elasticsearch (or OpenSearch) is a common choice for the search backend.

For an Elasticsearch tutorial, see the supplementary materials for Module 1.

## Fine-tuning vs RAG

Instead of retrieving documents at query time, you could fine-tune the LLM on your data.

Fine-tuning means taking a model's weights and adjusting them for your specific use case.

This works, but it has downsides:

- It requires special hardware (GPUs) and tools we don't cover in this course
- It's difficult to update when new data arrives - you don't want to retrain the model every time a new FAQ entry is added
- The LLM already has internal knowledge, but RAG gives it access to information it wasn't trained on

RAG is more flexible, cheaper, and works with any LLM. In practice, fine-tuning is rarely needed. I've never personally hit a case that required it. When I analyzed around 2,500 job descriptions for my AI engineering field guide, few asked for it. So focus on RAG first, and reach for fine-tuning only when you genuinely need it.

## Moving forward

ry these next steps:

- Try different prompts and see how the answers change
- Add more data sources to the knowledge base
- Experiment with different LLM models (GPT-4o, Claude, Gemini, local models via Ollama)
- Try Elasticsearch as a search backend